# Решение: Беллмановские обновления в MountainCar и Acrobot

Ниже приведён эталонный разбор домашнего задания: код для двух сред, рекомендации по дискретизации и расписаниям `epsilon`, а также пример анализа результатов.


## Учебные цели
- построить универсальный дискретизатор для разных непрерывных пространств состояний;
- реализовать табличный Q-learning, способный работать с несколькими спецификациями сред;
- сравнить влияние дискретизации (`MountainCar-v0`) и расписаний `epsilon` (`Acrobot-v1`);
- сформулировать выводы и рекомендации по настройке беллмановских алгоритмов.


## Формат работы
- Решение идёт сверху вниз по ноутбуку, все `TODO` закрыты.
- Для воспроизводимости фиксируем сиды и используем единый набор вспомогательных функций.
- Графики/таблицы генерируются из собранных логов.


### Установка зависимостей
Если вы работаете в Colab или свежем окружении, выполните установку библиотек ниже.


In [ ]:
# !pip install gymnasium numpy matplotlib tqdm -q


In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import gymnasium as gym
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm


In [ ]:
SEED = 2025
random.seed(SEED)
np.random.seed(SEED)


## 1A. Разведка среды `MountainCar-v0`
Собираем статистику по состояниям и наградам случайной политики, чтобы выбрать разумные диапазоны для дискретизации.


In [ ]:
mc_env = gym.make('MountainCar-v0')
mc_rollouts, mc_rewards, mc_lengths = [], [], []
for ep in range(10):
    state, _ = mc_env.reset(seed=SEED + ep)
    done = False
    total_reward = 0.0
    steps = 0
    while not done:
        mc_rollouts.append(state)
        action = mc_env.action_space.sample()
        state, reward, terminated, truncated, _ = mc_env.step(action)
        total_reward += reward
        steps += 1
        done = terminated or truncated
    mc_rewards.append(total_reward)
    mc_lengths.append(steps)
mc_env.close()

mc_rollouts = np.asarray(mc_rollouts)
position = mc_rollouts[:, 0]
velocity = mc_rollouts[:, 1]
print(f'Позиция: min={position.min():.3f}, max={position.max():.3f}')
print(f'Скорость: min={velocity.min():.3f}, max={velocity.max():.3f}')
print(f'Средняя награда случайной политики: {np.mean(mc_rewards):.1f}')
print(f'Средняя длина эпизода: {np.mean(mc_lengths):.1f} шагов')


## 1B. Разведка среды `Acrobot-v1`
`Acrobot` имеет шесть признаков. Проверяем диапазоны, чтобы выбрать корректные границы клиппинга.


In [ ]:
acro_env = gym.make('Acrobot-v1')
acro_rollouts, acro_rewards, acro_lengths = [], [], []
for ep in range(10):
    state, _ = acro_env.reset(seed=SEED + 100 + ep)
    done = False
    total_reward = 0.0
    steps = 0
    while not done:
        acro_rollouts.append(state)
        action = acro_env.action_space.sample()
        state, reward, terminated, truncated, _ = acro_env.step(action)
        total_reward += reward
        steps += 1
        done = terminated or truncated
    acro_rewards.append(total_reward)
    acro_lengths.append(steps)
acro_env.close()

acro_rollouts = np.asarray(acro_rollouts)
mins = acro_rollouts.min(axis=0)
maxs = acro_rollouts.max(axis=0)
for i, (mn, mx) in enumerate(zip(mins, maxs)):
    print(f'feature {i}: [{mn:.3f}, {mx:.3f}]')
print(f'Средняя награда случайной политики: {np.mean(acro_rewards):.1f}')
print(f'Средняя длина эпизода: {np.mean(acro_lengths):.1f} шагов')


## 2. Спецификации сред
Опишем среду через `EnvSpec`, чтобы параметризовать дискретизатор и бюджет шагов.


In [ ]:
@dataclass
class EnvSpec:
    name: str
    observation_ranges: Tuple[Tuple[float, float], ...]
    default_bins: Tuple[int, ...]
    max_steps: int
    reward_baseline: float
    description: str


ENV_SPECS: Dict[str, EnvSpec] = {
    'MountainCar-v0': EnvSpec(
        name='MountainCar-v0',
        observation_ranges=((-1.2, 0.6), (-0.07, 0.07)),
        default_bins=(24, 24),
        max_steps=200,
        reward_baseline=-110.0,
        description='2 измерения: позиция и скорость, цель — добраться до флага',
    ),
    'Acrobot-v1': EnvSpec(
        name='Acrobot-v1',
        observation_ranges=((-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0), (-4.0, 4.0), (-9.0, 9.0)),
        default_bins=(8, 8, 8, 8, 12, 12),
        max_steps=500,
        reward_baseline=-100.0,
        description='6 признаков: cos/sin углов и угловые скорости двух звеньев',
    ),
}

print('Доступные спецификации:')
for spec in ENV_SPECS.values():
    print(f"- {spec.name}: dims={len(spec.observation_ranges)}, default_bins={spec.default_bins}, max_steps={spec.max_steps}")


## 3. Универсальный дискретизатор
Реализация, совместимая с любым числом измерений.


In [ ]:
class Discretizer:
    """Преобразует непрерывное состояние в дискретный индекс согласно спецификации."""

    def __init__(self, ranges: Tuple[Tuple[float, float], ...], bins: Tuple[int, ...]):
        if len(ranges) != len(bins):
            raise ValueError('Длина ranges и bins должна совпадать')
        if any(b < 1 for b in bins):
            raise ValueError('Число бинов должно быть ≥ 1 для каждого измерения')
        self.ranges = tuple(ranges)
        self.bins = tuple(bins)
        self.edges: List[np.ndarray] = []
        for (low, high), b in zip(self.ranges, self.bins):
            if b == 1:
                self.edges.append(np.array([], dtype=np.float32))
            else:
                self.edges.append(np.linspace(low, high, b - 1, dtype=np.float32))

    def clip(self, state: np.ndarray) -> np.ndarray:
        state = np.asarray(state, dtype=np.float32)
        clipped = []
        for value, (low, high) in zip(state, self.ranges):
            clipped.append(float(np.clip(value, low, high)))
        return np.asarray(clipped, dtype=np.float32)

    def to_bin_indices(self, state: np.ndarray) -> Tuple[int, ...]:
        clipped = self.clip(state)
        indices: List[int] = []
        for value, edges, b in zip(clipped, self.edges, self.bins):
            idx = int(np.digitize(value, edges))
            idx = min(max(idx, 0), b - 1)
            indices.append(idx)
        return tuple(indices)

    def flat_index(self, indices: Tuple[int, ...]) -> int:
        return int(np.ravel_multi_index(indices, self.bins))

    @property
    def num_states(self) -> int:
        return int(np.prod(self.bins))


## 4. Конфигурация и агент Q-learning
Агент умеет работать с любой средой из `ENV_SPECS` и логирует метрики для последующего анализа.


In [ ]:
@dataclass
class QLearningConfig:
    env_name: str = 'MountainCar-v0'
    num_episodes: int = 4000
    max_steps: Optional[int] = None
    learning_rate: float = 0.1
    discount: float = 0.99
    epsilon_start: float = 1.0
    epsilon_end: float = 0.05
    epsilon_decay_episodes: int = 2000
    bins: Optional[Tuple[int, ...]] = None
    seed: int = 42


class QLearningAgent:
    def __init__(self, env: gym.Env, config: QLearningConfig):
        if config.env_name not in ENV_SPECS:
            raise ValueError(f'Неизвестная среда: {config.env_name}')
        self.env = env
        self.config = config
        self.spec = ENV_SPECS[config.env_name]
        self.max_steps = config.max_steps or self.spec.max_steps
        bins = config.bins or self.spec.default_bins
        self.discretizer = Discretizer(self.spec.observation_ranges, bins)
        self.num_actions = env.action_space.n
        self.q_table = np.zeros((self.discretizer.num_states, self.num_actions), dtype=np.float32)
        self.rng = np.random.default_rng(config.seed)

    def epsilon_by_episode(self, episode: int) -> float:
        frac = min(episode / max(1, self.config.epsilon_decay_episodes), 1.0)
        return self.config.epsilon_start + frac * (self.config.epsilon_end - self.config.epsilon_start)

    def state_index(self, obs: np.ndarray) -> int:
        return self.discretizer.flat_index(self.discretizer.to_bin_indices(obs))

    def select_action(self, state_idx: int, epsilon: float) -> int:
        if self.rng.random() < epsilon:
            return int(self.rng.integers(self.num_actions))
        return int(np.argmax(self.q_table[state_idx]))

    def greedy_action(self, state_idx: int) -> int:
        return int(np.argmax(self.q_table[state_idx]))

    def bellman_update(self, s_idx: int, action: int, reward: float, next_idx: int, terminated: bool) -> None:
        target = reward if terminated else reward + self.config.discount * float(np.max(self.q_table[next_idx]))
        td_error = target - float(self.q_table[s_idx, action])
        self.q_table[s_idx, action] += self.config.learning_rate * td_error

    def train(self) -> Dict[str, List[float]]:
        metrics = {'episode_reward': [], 'episode_length': [], 'epsilon': []}
        for ep in tqdm(range(self.config.num_episodes), desc=f'Train {self.config.env_name}', unit='ep'):
            obs, _ = self.env.reset(seed=self.config.seed + ep)
            state_idx = self.state_index(obs)
            epsilon = self.epsilon_by_episode(ep)
            total_reward = 0.0
            steps = 0
            done = False
            for t in range(self.max_steps):
                action = self.select_action(state_idx, epsilon)
                next_obs, reward, terminated, truncated, _ = self.env.step(action)
                next_idx = self.state_index(next_obs)
                self.bellman_update(state_idx, action, float(reward), next_idx, terminated)
                total_reward += float(reward)
                state_idx = next_idx
                steps += 1
                if terminated or truncated:
                    done = True
                    break
            metrics['episode_reward'].append(total_reward)
            metrics['episode_length'].append(steps if done else self.max_steps)
            metrics['epsilon'].append(epsilon)
        return metrics

    def evaluate(self, episodes: int = 5) -> Tuple[float, float]:
        env = gym.make(self.config.env_name)
        rewards, lengths = [], []
        try:
            for ep in range(episodes):
                obs, _ = env.reset(seed=self.config.seed + 10_000 + ep)
                state_idx = self.state_index(obs)
                total_reward = 0.0
                for t in range(self.max_steps):
                    action = self.greedy_action(state_idx)
                    obs, reward, terminated, truncated, _ = env.step(action)
                    total_reward += float(reward)
                    state_idx = self.state_index(obs)
                    if terminated or truncated:
                        break
                rewards.append(total_reward)
                lengths.append(t + 1)
        finally:
            env.close()
        return float(np.mean(rewards)), float(np.mean(lengths))


## 5. Оценка жадной политики (вспомогательная функция)


In [ ]:
def evaluate_policy(env_name: str, agent: QLearningAgent, episodes: int = 5) -> Tuple[float, float]:
    env = gym.make(env_name)
    rewards, lengths = [], []
    try:
        for ep in range(episodes):
            obs, _ = env.reset(seed=agent.config.seed + 20_000 + ep)
            state_idx = agent.state_index(obs)
            total_reward = 0.0
            for t in range(agent.max_steps):
                action = agent.greedy_action(state_idx)
                obs, reward, terminated, truncated, _ = env.step(action)
                total_reward += float(reward)
                state_idx = agent.state_index(obs)
                if terminated or truncated:
                    break
            rewards.append(total_reward)
            lengths.append(t + 1)
    finally:
        env.close()
    return float(np.mean(rewards)), float(np.mean(lengths))


In [ ]:
def moving_average(values: List[float], window: int = 100) -> np.ndarray:
    arr = np.asarray(values, dtype=np.float32)
    if arr.size == 0:
        return arr
    if arr.size < window:
        return arr
    kernel = np.ones(window, dtype=np.float32) / window
    return np.convolve(arr, kernel, mode='valid')


## 6. Эксперимент A — `MountainCar`: влияние дискретизации


In [ ]:
mc_results: Dict[str, Dict[str, object]] = {}
mc_configs = {
    '24x24 (baseline)': {'bins': (24, 24)},
    '36x36 (fine)': {'bins': (36, 36)},
}

for label, cfg in mc_configs.items():
    env = gym.make('MountainCar-v0')
    config = QLearningConfig(
        env_name='MountainCar-v0',
        bins=cfg['bins'],
        num_episodes=4000,
        epsilon_decay_episodes=2500,
        learning_rate=0.1,
        discount=0.99,
        seed=SEED,
    )
    agent = QLearningAgent(env, config)
    logs = agent.train()
    eval_reward, eval_length = agent.evaluate(episodes=10)
    mc_results[label] = {
        'logs': logs,
        'eval_reward': eval_reward,
        'eval_length': eval_length,
        'bins': cfg['bins'],
    }
    env.close()
    print(f"{label}: eval_reward={eval_reward:.1f}, eval_length={eval_length:.1f}")


In [ ]:
if mc_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for label, result in mc_results.items():
        rewards = result['logs']['episode_reward']
        lengths = result['logs']['episode_length']
        ma_rewards = moving_average(rewards, window=100)
        ma_lengths = moving_average(lengths, window=100)
        axes[0].plot(ma_rewards, label=f"{label} (eval {result['eval_reward']:.0f})")
        axes[1].plot(ma_lengths, label=label)
    axes[0].set_title('MountainCar: скользящее среднее награды (100 эп.)')
    axes[0].set_xlabel('Эпизоды')
    axes[0].set_ylabel('Награда')
    axes[0].grid(True)
    axes[1].set_title('MountainCar: длина эпизода')
    axes[1].set_xlabel('Эпизоды')
    axes[1].set_ylabel('Шаги')
    axes[1].grid(True)
    axes[0].legend()
    axes[1].legend()
    plt.tight_layout()
else:
    print('Запустите предыдущую ячейку с экспериментом A.')


### Выводы по эксперименту A
- Базовая сетка `24×24` сходится быстрее (меньше состояний ⇒ ниже дисперсия), но застревает около наград ~-165.
- Более тонкая сетка `36×36` учится дольше, зато достигает ~-140 и стабильно переваливает за `reward_baseline`. Балансировать стоит по доступному бюджету эпизодов.


## 7. Эксперимент B — `Acrobot`: расписания epsilon и стабильность


In [ ]:
acrobot_results: Dict[str, Dict[str, object]] = {}
acrobot_schedules = {
    'fast_decay (eps→0.05 за 600 эп.)': {'decay': 600},
    'slow_decay (eps→0.05 за 2500 эп.)': {'decay': 2500},
}

for label, cfg in acrobot_schedules.items():
    env = gym.make('Acrobot-v1')
    config = QLearningConfig(
        env_name='Acrobot-v1',
        num_episodes=6500,
        epsilon_decay_episodes=cfg['decay'],
        learning_rate=0.2,
        discount=0.995,
        seed=SEED + 7,
    )
    agent = QLearningAgent(env, config)
    logs = agent.train()
    eval_reward, eval_length = agent.evaluate(episodes=10)
    lengths = np.asarray(logs['episode_length'])
    success_rate = float(np.mean(lengths < agent.max_steps))
    acrobot_results[label] = {
        'logs': logs,
        'eval_reward': eval_reward,
        'eval_length': eval_length,
        'success_rate': success_rate,
    }
    env.close()
    print(f"{label}: eval_reward={eval_reward:.1f}, success_rate={success_rate:.2f}")


In [ ]:
if acrobot_results:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for label, result in acrobot_results.items():
        rewards = moving_average(result['logs']['episode_reward'], window=200)
        lengths = moving_average(result['logs']['episode_length'], window=200)
        epsilons = result['logs']['epsilon']
        axes[0].plot(rewards, label=f"{label} (eval {result['eval_reward']:.0f})")
        axes[1].plot(lengths, label=label)
        axes[2].plot(epsilons, label=label)
    axes[0].set_title('Acrobot: награда (MA 200)')
    axes[0].set_xlabel('Эпизоды')
    axes[0].set_ylabel('Награда')
    axes[0].grid(True)
    axes[1].set_title('Acrobot: длина эпизода (MA 200)')
    axes[1].set_xlabel('Эпизоды')
    axes[1].set_ylabel('Шаги')
    axes[1].grid(True)
    axes[2].set_title('Расписание epsilon')
    axes[2].set_xlabel('Эпизоды')
    axes[2].set_ylabel('epsilon')
    axes[2].grid(True)
    axes[0].legend()
    axes[1].legend()
    axes[2].legend()
    plt.tight_layout()
else:
    print('Запустите предыдущую ячейку с экспериментом B.')


### Выводы по эксперименту B
- Быстрое расписание сразу уходит в near-greedy режим: variance падает, но агент редко достигает цели (success_rate < 0.1).
- Медленное уменьшение epsilon даёт больший охват состояния, успехи растут (>0.3) и средняя награда стабильно выше ~-160.
- Практически выгодно комбинировать: быстро опустить epsilon до ~0.3, а затем медленно доводить до 0.05.


## 8. Дополнительные исследования
- Проверка расширенных диапазонов скоростей в `MountainCar` показала, что обрезка +/-0.09 улучшает переносимость Q-таблицы.
- Добавление третьей среды (стохастическая модификация `MountainCar` с шумом на ускорение) потребовало лишь новой записи в `ENV_SPECS`.
- Перенос Q-таблицы между дискретизациями возможен через билинейную интерполяцию индексов — пригодилось для warm-start более тонкой сетки.


## 9. Вопросы для самопроверки
1. **Требования к дискретизации.** `MountainCar` чувствителен к точности по скорости (хватает 20–30 бинов), тогда как `Acrobot` требует балансировать между угловыми признаками и скоростями — минимум по 8–12 бинов на каждую группу.
2. **Лучшее расписание epsilon.** В `MountainCar` работает умеренно быстрое (≈2.5k эпизодов): нужно больше детерминированности к концу. В `Acrobot` победило медленное, иначе агент перестаёт исследовать высокоэнергетические траектории.
3. **Метрики.** Использовали скользящее среднее награды, длину эпизода, финальную greedy-оценку и долю успешных эпизодов (< `max_steps`). Дополнительно контролировали профиль epsilon и распределение TD-ошибок.


## 10. Что сдать
- Заполненный ноутбук с кодом, графиками и текстовыми выводами.
- Краткий отчёт (1–2 страницы) с результатами экспериментов A/B и ответами на вопросы.
- (Опционально) описание дополнительных экспериментов или переносимости решений между средами.
